# ODIN Phasemeter Data: Loading Zipped Moku Exports

Lab ring-downs from the ODIN test stands arrive as **zipped Moku:Pro Phasemeter CSV
exports** (`data/ODIN/*.csv.zip`): a `%`-comment header block followed by five columns
per input, sampled at about 149 Hz, running for hours to days. Uncompressed these are
1–4 GB per file, which is why they stay zipped and why nothing here loads a whole
record.

This notebook covers the part that is specific to those files — getting a usable
`(t, phase)` pair out of one — and then hands off to the analysis you already know
from [`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb).

**Prerequisites**

- [`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb) for the real-data Q
  workflow (`Q_selected`, `Q_demod`, drift diagnostics). This notebook does not
  re-explain those fields.
- `mokutools` must be importable. It is not a dependency of `ringdownanalysis`:
  `pip install git+https://github.com/mdovale/mokutools`.
- At least one `.csv.zip` under `data/ODIN/`. That directory is not tracked in git
  (`data/` is ignored), so the files must be copied in locally.

Runtime is a few tens of seconds: one capped window is loaded and analyzed.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mokutools.phasemeter import MokuPhasemeterObject

from ringdownanalysis import RingDownAnalyzer, configure_logging, plots
from ringdownanalysis.estimators import DFTFrequencyEstimator

plots.apply_plotting_style()
configure_logging(level=logging.WARNING)

## 1. Which loader for which file

| File kind | Loader | Notes |
| --- | --- | --- |
| `.csv`, `.mat` (Moku:Lab layout) | `RingDownDataLoader` / `RingDownAnalyzer.analyze_file()` | Reads time from column 0 and phase from column 3, detrends, shifts time to start at 0. Format spec: `docs/data_format.md`. |
| `.csv.zip` (Moku:Pro export) | `mokutools.phasemeter.MokuPhasemeterObject` | Parses the `%` header for sampling rate, date, and channel count; names all five columns per input; reads a row range without expanding the archive. |

`RingDownDataLoader` cannot be pointed at these archives: the layout is different (five
columns per input rather than the fixed four-column Moku:Lab layout), the sampling rate
lives in the header rather than being implied, and a windowed read is not optional at
this file size. So the split of labour is: `mokutools` produces arrays, then the
`ringdownanalysis` pipeline analyzes arrays via `analyze_array()`.

One header caveat: the `%` block is what carries the acquisition rate. A record that has
been re-exported or post-processed into a bare `time,1_set_freq,...` CSV will fail with
`ValueError: No header lines detected`, and there is no sampling rate to recover. Keep
the original exports.

In [ ]:
DATA_DIR = (Path("..") / "data" / "ODIN").resolve()

# Each record needs its own channel and window: the ring-down sits on a particular
# input, and the analysis window has to contain the release. These three are known
# good; other records under data/ODIN/ follow the same pattern with their own values.
CANDIDATES = [
    {
        "name": "20260321_EDU_R1.csv.zip",
        "label": "EDU R1, pre-vibration",
        "channels": ("1_cycles",),
        "duration_s": 2 * 3600.0,
    },
    {
        "name": "20260511_SN2.csv.zip",
        "label": "ODIN SN2, differential inputs 1-2",
        "channels": ("1_cycles", "2_cycles"),
        "duration_s": 4 * 3600.0,
    },
    {
        "name": "20260512_SN2.csv.zip",
        "label": "ODIN SN2, differential inputs 1-2",
        "channels": ("1_cycles", "2_cycles"),
        "duration_s": 4 * 3600.0,
    },
]

print(f"Looking in {DATA_DIR}")
record = None
for candidate in CANDIDATES:
    path = DATA_DIR / candidate["name"]
    available = path.is_file()
    print(f"  [{'found  ' if available else 'missing'}] {candidate['name']}  ({candidate['label']})")
    if available and record is None:
        record = dict(candidate, path=path)

if record is None:
    expected = "\n  ".join(str(DATA_DIR / c["name"]) for c in CANDIDATES)
    raise FileNotFoundError(
        "No ODIN phasemeter export found. Expected one of:\n  "
        + expected
        + "\n\nThese are multi-GB Moku:Pro exports and are not tracked in git; copy "
        "one into data/ODIN/ to run this notebook."
    )

print(f"\nUsing {record['name']} - {record['label']}")

## 2. `start_time` is an offset, not a timestamp

This is the mistake that silently ruins an ODIN analysis. From
`docs/data_format.md` § "Loading via mokutools":

> **`start_time` is an offset, not an absolute time.** The `start_time` argument
> selects data relative to the *first sample of the file*. Passing an absolute
> timestamp silently selects the wrong window.

The confusion is easy to fall into because the `time` column of these exports does
**not** start at zero — the phasemeter had already been logging when the record was
split, so the first timestamp is some hundreds or thousands of seconds. Reading that
first timestamp and feeding it back as `start_time` throws away exactly that many
seconds of data, and no error is raised.

The two loads below make the arithmetic visible: `start_time` is converted to a row
index as `int(start_time * fs)` counted from the first data row, so the first timestamp
you get back is `first_timestamp_in_file + start_time`.

In [ ]:
probe_0 = MokuPhasemeterObject(filename=str(record["path"]), start_time=0.0, duration=60.0)
first_timestamp = float(probe_0.df["time"].iloc[0])
print(f"fs = {probe_0.fs:.3f} Hz, acquisition date = {probe_0.date}")
print(f"start_time=0    -> time column starts at {first_timestamp:.2f} s")

probe_600 = MokuPhasemeterObject(filename=str(record["path"]), start_time=600.0, duration=60.0)
print(f"start_time=600  -> time column starts at {float(probe_600.df['time'].iloc[0]):.2f} s")
print(f"                   i.e. {first_timestamp:.2f} + 600 s: the offset skips 600 s of data")

print(
    f"\nPassing start_time={first_timestamp:.2f} (the first timestamp) would discard the "
    f"first {first_timestamp:.0f} s of the record for no reason."
)

## 3. Loading the analysis window

Three choices to make explicit:

- **Window.** `start_time=0` and a capped `duration` keep this notebook interactive.
  For the EDU record the decay lasts several hours; two hours covers roughly two decay
  times, which is enough for the incoherent estimator. The window must contain the
  release — a window that starts after the resonator has settled back to its
  ambient-driven level yields `Q_demod_status="plateau_dominated"` and no `Q_selected`.
- **Channel.** The ring-down lives on one input (`1_cycles` here) or on a differential
  pair (`1_cycles - 2_cycles` for the SN2 records, cancelling common-mode phase between
  the two beat notes). `*_cycles` columns are phase in cycles, which is the unit the
  pipeline expects; `mokutools` also derives `*_phase` in radians.
- **Time base.** `pm.df["time"]` carries the file's absolute timestamps, so subtract the
  first sample before handing it to the analyzer. Passing the unshifted column makes the
  crop logic and every time-referenced diagnostic wrong by the offset.

In [ ]:
pm = MokuPhasemeterObject(
    filename=str(record["path"]),
    start_time=0.0,
    duration=record["duration_s"],
)

columns = record["channels"]
if len(columns) == 1:
    phase_cycles = pm.df[columns[0]].to_numpy()
    channel_label = columns[0]
else:
    phase_cycles = pm.df[columns[0]].to_numpy() - pm.df[columns[1]].to_numpy()
    channel_label = f"{columns[0]} - {columns[1]}"

t_absolute = pm.df["time"].to_numpy()
t = t_absolute - t_absolute[0]

print(f"file        : {record['name']}")
print(f"channel     : {channel_label}")
print(f"fs          : {pm.fs:.3f} Hz, {pm.nchan} phasemeter channels in file")
print(f"window      : {pm.duration:.0f} s ({pm.ndata} samples), rows {pm.start_row}-{pm.end_row}")
print(f"absolute t  : {t_absolute[0]:.0f} s to {t_absolute[-1]:.0f} s")
print(f"phase range : {phase_cycles.min():.1f} to {phase_cycles.max():.1f} cycles")

In [ ]:
step = max(1, len(t) // 20_000)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[::step] / 3600.0, phase_cycles[::step], linewidth=0.8, alpha=0.85)
ax.set_xlabel("Time since window start (h)")
ax.set_ylabel("Phase (cycles)")
ax.set_title(f"{record['label']} - {channel_label} (raw)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Running the pipeline on the window

Two arguments matter for records like this one:

- **`detrend="constant"`** removes the mean, matching what `RingDownDataLoader` does for
  file loading. The raw plot above shows why: the oscillation is not centred on zero and
  its centre wanders slowly. Leaving that offset in biases the amplitude and offset
  parameters of every fit.
- **`DFTFrequencyEstimator(f_min=1.0)`** restricts the DFT peak search to above 1 Hz.
  Moku phase records carry strong sub-Hz baseline wander; where that wander is
  comparable to the ring-down — whole-record loads, or windows that are mostly plateau —
  an unrestricted search can return the wander instead of the resonance near 7.7 Hz. On
  a capped, decay-dominated window like this one it usually makes no difference, so the
  cell prints the estimate both ways rather than asserting it. Supplying the band is
  cheap insurance and is what the ODIN notebooks do (see
  [`0.4_frequency-estimation.ipynb`](0.4_frequency-estimation.ipynb) § 3).

In [ ]:
centered = phase_cycles - phase_cycles.mean()
for f_min in (0.0, 1.0):
    f_peak = DFTFrequencyEstimator(window="rect", f_min=f_min).estimate(centered, pm.fs)
    print(f"DFT peak with f_min={f_min:.1f} Hz: {f_peak:.6f} Hz")

analyzer = RingDownAnalyzer(dft_estimator=DFTFrequencyEstimator(window="rect", f_min=1.0))
result = analyzer.analyze_array(t=t, data=phase_cycles, detrend="constant")
print()


def fmt(value, spec=".4g"):
    """Format a possibly-missing estimate."""
    return format(value, spec) if value is not None and np.isfinite(value) else "-"


ci = result["Q_demod_ci95"]
print("--- recommended answer ---")
print(
    f"Q_selected  : {fmt(result['Q_selected'], '.4e')}"
    f"  [source {result['Q_selected_source']}, regime {result['Q_selected_regime']},"
    f" status {result['Q_selected_status']}]"
)
if result["Q_selected_reasons"]:
    print(f"  reasons   : {', '.join(result['Q_selected_reasons'])}")

print("\n--- segmented demodulation (drift-immune) ---")
print(f"Q_demod     : {fmt(result['Q_demod'], '.4e')} ({result['Q_demod_status']})")
print(f"  95% CI    : [{fmt(ci[0], '.4e')}, {fmt(ci[1], '.4e')}]" if ci else "  95% CI    : -")
print(f"tau_demod   : {fmt(result['tau_demod'], '.0f')} s over {result['Q_demod_n_segments']} segments")
print(f"f_demod     : {fmt(result['f_demod'], '.6f')} Hz")

print("\n--- why the coherent fits are not used ---")
print(f"drift       : {1e3 * result['Q_demod_drift_hz']:+.3f} mHz across the decay")
print(f"coherence_ratio |drift| x tau : {result['coherence_ratio']:.2f}  (coherent fits need < 0.01)")
print(f"plateau     : {fmt(result['Q_demod_plateau_amplitude'], '.1f')} cycles, detected={result['Q_demod_plateau_detected']}")
print(f"Q_profile   : {fmt(result['Q_profile'], '.4e')} ({result['Q_profile_status']}) {result['Q_profile_reasons']}")
print(f"Q_envelope  : {fmt(result['Q_envelope'], '.4e')}  (incoherent, biased high near the plateau)")
print(f"f_nls       : {result['f_nls']:.6f} Hz, f_dft: {result['f_dft']:.6f} Hz")

## 5. Reading the diagnostics

The numbers above are the real-data story from
[`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb), now on a file you
loaded yourself:

- `Q_selected` comes from `Q_demod`, because the record is classified
  `long_or_drifting`.
- The frequency drifts by of order 1 mHz over the decay. Multiplied by $\tau$ that is a
  coherence ratio in the single or double digits — two to three orders of magnitude past
  the 0.01 tolerance of a phase-coherent fit — so `Q_profile` is gated out no matter how
  tight its interval looks.
- The resonator never decays to zero: ambient excitation holds it near the floor the
  estimator fits (the dotted line in the left panel), and the last segments of this
  window visibly flatten toward it. `Q_demod_plateau_detected` is `False` here because
  that flattening is still marginal; a longer window makes it unambiguous, and a window
  containing nothing but plateau is unusable for Q
  (`Q_demod_status="plateau_dominated"`). `Q_envelope` runs a few percent above
  `Q_demod`, the direction expected once a window reaches toward the floor.

The figure below is the evidence for both statements — the demodulated amplitude with
its fitted decay and plateau, and the per-segment frequency that measures the drift.

In [ ]:
demod = result["Q_demod_result"]
mask = demod.decay_mask

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
ax.semilogy(demod.t_mid / 3600.0, demod.amplitude, ".", ms=4, alpha=0.6, label="segment amplitude")
t_fit = demod.t_mid[mask]
ax.semilogy(
    t_fit / 3600.0,
    np.exp(demod.log_slope * t_fit + demod.log_intercept),
    "-",
    linewidth=2,
    label=f"fit: tau = {demod.tau:.0f} s, Q = {demod.Q:.3g}",
)
if demod.plateau_amplitude is not None:
    ax.axhline(
        demod.plateau_amplitude,
        linestyle=":",
        linewidth=1.2,
        label=f"plateau = {demod.plateau_amplitude:.0f} cycles",
    )
ax.set_xlabel("Time since window start (h)")
ax.set_ylabel("Segment amplitude (cycles)")
ax.set_title("Demodulated decay and driven plateau")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(demod.t_mid[mask] / 3600.0, 1e3 * (demod.f_seg[mask] - demod.f_seg[mask][0]), ".", ms=4)
ax.set_xlabel("Time since window start (h)")
ax.set_ylabel("Frequency change (mHz)")
ax.set_title(f"Frequency drift: coherence ratio = {result['coherence_ratio']:.2f}")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Troubleshooting and next steps

| Symptom | Cause | Fix |
| --- | --- | --- |
| `ValueError: No header lines detected` | the export lost its `%` header block | use the original export; the header carries the sampling rate |
| `Q_demod_status="plateau_dominated"`, `Q_selected=None` | the window missed the release | move `start_time` earlier or lengthen `duration` |
| Frequency estimate far from the expected resonance | DFT locked onto baseline wander | pass `DFTFrequencyEstimator(f_min=...)` |
| Diagnostics inconsistent with the plot | absolute timestamps passed as `t` | subtract `t[0]`, and remember `start_time` is an offset |
| Q changes a lot with the window | genuine amplitude-dependent damping, or a window past the plateau | `RingDownAnalyzer.q_sensitivity()`, and compare Q at matched amplitude |

This notebook deliberately analyzes a single capped window. For the wider workflows:

- Many files at once, with consistency and summary tables:
  [`0.2_batch-analysis.ipynb`](0.2_batch-analysis.ipynb)
- Whole-record analysis of these same EDU files, amplitude-resolved Q, and the
  nonlinear-damping model:
  [`20260819_EDU_PreVibe_vs_PostVibe_Demod_Comparison.ipynb`](20260819_EDU_PreVibe_vs_PostVibe_Demod_Comparison.ipynb)
  and the full investigation in
  [`20260818_EDU_PreVibe_vs_PostVibe_RingDown.ipynb`](20260818_EDU_PreVibe_vs_PostVibe_RingDown.ipynb)

### Next steps

- Statistical foundations behind the estimators: [`0.6_monte-carlo-crlb.ipynb`](0.6_monte-carlo-crlb.ipynb)
- Why the coherent estimators are gated here: [`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb)
  and README § "Which Q should I trust?"